In [51]:
from flax import nnx
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
sys.path.append("..")

from qnrl.algorithms.algorithm import Algorithm

import wandb
api = wandb.Api()

In [52]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [53]:
project_id = "qnrl"

In [54]:
import yaml
import ast
from qnrl.serialization import hash_serializable_object

def get_run_config(config_file: Path):
    with open(str(config_file), 'r') as f:
        config = yaml.safe_load(f)
    return config

def get_run_metadata(metadata_file: Path):
    # Load metadata as JSON dictionary.
    with open(str(metadata_file), 'r') as f:
        metadata = json.load(f)
    return metadata

def get_run_config_dict_from_metadata(metadata_file: Path):
    """Compute run config hash from original run program command string."""
    metadata = get_run_metadata(metadata_file)
    
    # Get config string from run program command arguments.
    # Format is: --config="{...}"
    cfg_str = metadata['args'][1].split("--config=")[1]
    # Convert this string to dictionary via literal evaluation.
    cfg_dict = ast.literal_eval(cfg_str)
    return cfg_dict

In [55]:
# Load algorithm config for sweep and run.
# sweep_id = "1nlfftsy"
# run_id = "5ti67g7h"

meta = {
    "sweep_id": "4ykq4hvw",
    "run_id": "atroi3e3",
}
dir_run = root_sweeps/f"sweep-{meta['sweep_id']}"/f"run-{meta['run_id']}"

cfg = get_run_config_dict_from_metadata(dir_run/"wandb-metadata.json")
cfg_hash, _ = hash_serializable_object(cfg) # Compute hash for config object.
alg = Algorithm.from_dict(cfg)

nnx.display(alg)

In [56]:
# Verify run configuration hash matches hash of algorithm configuration object.
run_config = get_run_config(dir_run/"config.yaml")
run_config['config_hash']['value'] == cfg_hash

True

In [57]:
# Display initial state of model parameters.
nnx.display(nnx.state(alg.model))

In [58]:
# Load parameters from model save.
model_dirs = {Path(root).stem: Path(root) for root in glob.glob(str(dir_run/"models"/"*"))}
alg.load(model_dirs)
nnx.display(nnx.state(alg.model))